# Baseline 1: Traditional Machine Learning (Naive Bayes)

This notebook implements the first baseline model for the context-aware toxicity detection project. We use a traditional **Multinomial Naive Bayes** classifier paired with **TF-IDF (Term Frequency-Inverse Document Frequency)** vectorization. This serves as a computational baseline to measure the improvements brought by Transformer-based architectures later in the study.

In [4]:
# Import necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

## 1. Data Loading and Preprocessing
Loading the Jigsaw dataset from the `data/` directory. For this baseline, we will isolate the `comment_text` (features) and the `toxic` column (target). Missing values (NaN) are dropped to ensure clean data.

In [5]:
import pandas as pd

print("Loading data...")

# Load the dataset
df = pd.read_csv('../data/train.csv')

# Select only the relevant columns and drop rows with missing values
df = df[['comment_text', 'toxic']].dropna()

# Display the first few rows of the dataframe
df.head()

Loading data...


,comment_text,toxic
0,Explanation\nWhy the edits made under my usern...,0
1,D'aww! He matches this background colour I'm s...,0
2,"Hey man, I'm really not trying to edit war. It...",0
3,"""\nMore\nI can't make any real suggestions on ...",0
4,"You, sir, are my hero. Any chance you remember...",0


## 2. Train-Test Split
The dataset is split into training (80%) and testing (20%) sets to evaluate the model's performance on unseen data.

In [6]:
print("Splitting data...")
X_train, X_test, y_train, y_test = train_test_split(
    df['comment_text'], 
    df['toxic'], 
    test_size=0.2, 
    random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

Splitting data...
Training samples: 127656
Testing samples: 31915


## 3. Feature Extraction (TF-IDF)
Since Machine Learning models cannot process raw text, we convert the comments into numerical vectors using TF-IDF. We limit the vocabulary to the top 10,000 features and remove common English stop words to reduce noise.

In [7]:
print("Performing TF-IDF vectorization...")
vectorizer = TfidfVectorizer(max_features=10000, stop_words='english')

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

Performing TF-IDF vectorization...


## 4. Model Training
Training a Multinomial Naive Bayes classifier on the vectorized training data. Naive Bayes is chosen for its efficiency and standard use as a baseline in text classification tasks.

In [8]:
print("Training the model...")
model = MultinomialNB()
model.fit(X_train_vec, y_train)
print("Training complete.")

Training the model...
Training complete.


## 5. Evaluation
Generating predictions on the test set and printing the classification report. The core metrics to observe here are Precision, Recall, and the F1-score for the 'Toxic' class.

In [9]:
print("Generating predictions...")
y_pred = model.predict(X_test_vec)

print("\n--- INITIAL BASELINE RESULTS ---")
print(classification_report(y_test, y_pred, target_names=['Non-Toxic', 'Toxic']))

Generating predictions...

--- INITIAL BASELINE RESULTS ---
              precision    recall  f1-score   support

   Non-Toxic       0.95      1.00      0.97     28859
       Toxic       0.92      0.51      0.66      3056

    accuracy                           0.95     31915
   macro avg       0.94      0.75      0.82     31915
weighted avg       0.95      0.95      0.94     31915



## 6. Addressing the Accuracy Paradox (Undersampling)

As seen in the initial baseline results, the model achieved a high accuracy (95%) but struggled significantly with the 'Toxic' class, capturing only 51% of actual toxic comments (Recall: 0.51). This is a classic example of the **accuracy paradox** caused by severe class imbalance. 

To ensure our model genuinely learns to distinguish between toxic and non-toxic language—rather than simply guessing the majority class—we will manipulate the dataset using **undersampling**. By randomly reducing the majority class (Non-Toxic) to match the exact size of the minority class (Toxic), we create a balanced 50/50 dataset. We will then retrain and re-evaluate the Naive Bayes model on this balanced data.

In [10]:
from sklearn.utils import resample

print("Addressing class imbalance via undersampling...")

# 1. Separate majority and minority classes using the already loaded 'df'
df_non_toxic = df[df['toxic'] == 0]
df_toxic = df[df['toxic'] == 1]

# 2. Downsample the majority class (non-toxic) to match the minority class (toxic)
df_non_toxic_downsampled = resample(df_non_toxic, 
                                    replace=False,           # Sample without replacement
                                    n_samples=len(df_toxic), # Match minority class size
                                    random_state=42)         # Ensure reproducibility

# 3. Combine minority class with downsampled majority class
df_balanced = pd.concat([df_toxic, df_non_toxic_downsampled])

# Shuffle the balanced dataset to ensure random distribution
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

# Verify the new class distribution
print("\nClass Distribution After Undersampling:")
print(df_balanced['toxic'].value_counts(normalize=True))

# 4. Feature Extraction and Train-Test Split for Balanced Data
X_bal = df_balanced['comment_text']
y_bal = df_balanced['toxic']

X_train_bal, X_test_bal, y_train_bal, y_test_bal = train_test_split(X_bal, y_bal, test_size=0.2, random_state=42)

# We can re-initialize the vectorizer to fit specifically on the balanced vocabulary
vectorizer_bal = TfidfVectorizer(max_features=10000, stop_words='english')
X_train_bal_vec = vectorizer_bal.fit_transform(X_train_bal)
X_test_bal_vec = vectorizer_bal.transform(X_test_bal)

# 5. Train the Model on Balanced Data
print("\nTraining the model on the balanced dataset...")
model_bal = MultinomialNB()
model_bal.fit(X_train_bal_vec, y_train_bal)

# 6. Evaluate the Balanced Model
print("Generating predictions...")
y_pred_bal = model_bal.predict(X_test_bal_vec)

print("\n--- BALANCED BASELINE RESULTS ---")
print(classification_report(y_test_bal, y_pred_bal, target_names=['Non-Toxic', 'Toxic']))

Addressing class imbalance via undersampling...

Class Distribution After Undersampling:
toxic
0    0.5
1    0.5
Name: proportion, dtype: float64

Training the model on the balanced dataset...
Generating predictions...

--- BALANCED BASELINE RESULTS ---
              precision    recall  f1-score   support

   Non-Toxic       0.89      0.87      0.88      3039
       Toxic       0.88      0.89      0.88      3079

    accuracy                           0.88      6118
   macro avg       0.88      0.88      0.88      6118
weighted avg       0.88      0.88      0.88      6118

